# Catálogo de mercado v2: IA ajustada como classificador, com a quantidade (GPU T4 do Kaggle)

**Português** (English summary at the end). Repositório: https://github.com/Bueno-00/catalogo-de-mercado

No v1 o LoRA gerava o nome da categoria como texto (uma palavra por vez, 0,7 s por produto um por vez). Aqui ele
ganha uma **cabeça de classificação**: uma passada pelo modelo dá a probabilidade de cada categoria. A
quantidade lida e normalizada vai no texto (`Leite Nestlé 34 g | quantidade: 34 g`). Modelos:

1. **BERTimbau base** (110 M de parâmetros, ajuste completo), com e sem a quantidade;
2. **Qwen2.5-1.5B + LoRA** com cabeça de classificação;
3. extra: **Qwen2.5-3B + LoRA**.

Mesmo dataset privado do v1 (divisão por marca). A época de cada modelo é escolhida pela **validação**. As
métricas do teste são calculadas fora daqui, pelo mesmo script dos outros caminhos.

In [ ]:
import os, re, gc, time, json, math, random, subprocess, sys, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

LOCAL = os.environ.get("CATALOGO_LOCAL") == "1"
SEED = 42
LIMITE = int(os.environ.get("CATALOGO_LIMITE", "0"))
MAX_LEN = 64

def achar_dados():
    for raiz in [Path("/kaggle/input"), Path(os.environ.get("CATALOGO_DADOS", "data/split"))]:
        if raiz.exists():
            achados = sorted(raiz.rglob("teste.parquet"))
            if achados:
                return achados[0].parent
    raise FileNotFoundError("teste.parquet não encontrado")

DADOS = achar_dados()
SAIDA = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("CATALOGO_SAIDA", ".tmp/kaggle-v2-local"))
SAIDA.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
import torch
torch.manual_seed(SEED)
GPU = torch.cuda.is_available()
DISP = "cuda" if GPU else "cpu"
import transformers
if Path("/kaggle/working").exists():
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
try:
    import peft
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft>=0.13"], check=True)
    import peft
print(f"dados={DADOS} | gpu={GPU} {torch.cuda.get_device_name(0) if GPU else ''} | torch {torch.__version__} | "
      f"transformers {transformers.__version__} | peft {peft.__version__} | cpus={os.cpu_count()}")

## Quantidade (o mesmo código de `src/catalogo/quantidade.py`)

In [ ]:
# --- copiado de src/catalogo/quantidade.py ---


from __future__ import annotations

import re
import unicodedata

# número com vírgula ou ponto (decimal ou milhar), um "x" opcional de pacote antes, e a unidade
_NUM = r"\d+(?:[.,]\d+)?"
_UNIDADE = (r"kg|kgs|kilos?|quilos?|g|gr|grs|grama|gramas|mg|ml|mls|cl|l|lt|lts|litro|litros|oz"
            r"|un|und|unds|unid|unids|unidade|unidades|uni|cap|caps|capsula|capsulas|sache|saches|sachet|sachets")
_QTD = re.compile(r"(?:(\d{1,3})\s*x\s*)?(" + _NUM + r")\s*(" + _UNIDADE + r")(?![a-z])")
_PACOTE = re.compile(r"(?<![a-z0-9])(\d{1,3})\s*x(?![a-z])")

MASSA = {"kg": 1000.0, "kgs": 1000.0, "kilo": 1000.0, "kilos": 1000.0, "quilo": 1000.0, "quilos": 1000.0,
         "g": 1.0, "gr": 1.0, "grs": 1.0, "grama": 1.0, "gramas": 1.0, "mg": 0.001, "oz": 28.35}
VOLUME = {"ml": 1.0, "mls": 1.0, "cl": 10.0, "l": 1000.0, "lt": 1000.0, "lts": 1000.0, "litro": 1000.0, "litros": 1000.0}
# faixas (limite de cima de cada uma); a última vai até o infinito
FAIXAS = {"massa": [20, 45, 90, 180, 350, 700, 1500], "volume": [250, 400, 700, 1200, 2200], "unid": [1, 6, 20]}
NOME_TIPO = {"massa": "m", "volume": "v", "unid": "u"}
# palavras que não dizem o que o produto é (ficam fora da combinação palavra + faixa)
VAZIAS = {"de", "da", "do", "das", "dos", "com", "sem", "em", "e", "a", "o", "para", "sabor", "tipo", "zero",
          "light", "diet", "integral", "tradicional", "original", "novo", "nova", "mini", "kit", "pack", "the"}


def _ascii(s: str) -> str:
    return unicodedata.normalize("NFKD", str(s or "")).encode("ascii", "ignore").decode().lower()


def _numero(txt: str, unidade: str) -> float:
    """"1,5" = 1.5; "1.000" com g ou ml = 1000 (milhar); "0,500" com kg = 0.5."""
    if re.fullmatch(r"\d+[.,]\d{3}", txt) and not txt.startswith("0") and unidade in ("g", "gr", "grs", "ml", "mg"):
        return float(txt.replace(".", "").replace(",", ""))
    return float(txt.replace(",", "."))


def ler(texto: str) -> dict:
    """Última quantidade do texto (a do campo quantidade vem no fim da linha). Peso em gramas, volume em ml.

    >>> ler("Leite Nestlé 34 g")["valor"], ler("Cerveja 6x350ml")["pacote"], ler("Arroz 5kg")["valor"]
    (34.0, 6, 5000.0)
    """
    s = _ascii(texto)
    achados = list(_QTD.finditer(s))
    if not achados:
        return {"tipo": "nenhum", "valor": 0.0, "pacote": 1, "faixa": -1}
    m = achados[-1]
    unidade = m.group(3)
    valor = _numero(m.group(2), unidade)
    if unidade in MASSA:
        tipo, valor = "massa", valor * MASSA[unidade]
    elif unidade in VOLUME:
        tipo, valor = "volume", valor * VOLUME[unidade]
    else:
        tipo = "unid"
    pacote = int(m.group(1)) if m.group(1) else 1
    if pacote == 1:
        p = _PACOTE.search(s)
        if p and tipo != "unid":
            pacote = int(p.group(1))
    faixa = sum(valor > lim for lim in FAIXAS[tipo])
    return {"tipo": tipo, "valor": round(valor, 3), "pacote": pacote, "faixa": faixa}


def palavras_chave(texto_norm: str, n: int = 2) -> list[str]:
    """As primeiras palavras do nome que dizem o que o produto é (só letras, 3 ou mais, fora as vazias)."""
    out = []
    for w in texto_norm.split():
        if len(w) >= 3 and w.isalpha() and w not in VAZIAS:
            out.append(w)
            if len(out) == n:
                break
    return out


def sinais(texto: str, texto_norm: str) -> str:
    """Palavras extras para o modelo clássico: tipo, faixa, pacote e a combinação palavra + faixa.

    >>> sinais("Leite Nestlé 34 g", "leite nestle 34 g")
    'qt_m qf_m1 qx_leite_m1 qx_nestle_m1'
    """
    q = ler(texto)
    if q["tipo"] == "nenhum":
        return "qt_nada " + " ".join(f"qx_{w}_nada" for w in palavras_chave(texto_norm, 1))
    t = NOME_TIPO[q["tipo"]]
    f = f"{t}{q['faixa']}"
    out = [f"qt_{t}", f"qf_{f}"]
    if q["pacote"] > 1:
        out.append("qp_pacote")
    out += [f"qx_{w}_{f}" for w in palavras_chave(texto_norm)]
    return " ".join(out)


def texto_com_quantidade(texto: str) -> str:
    """Para as IAs: a linha do produto + a quantidade normalizada ("| quantidade: 34 g")."""
    q = ler(texto)
    if q["tipo"] == "nenhum":
        return str(texto)
    v = q["valor"] if q["valor"] < 10 else round(q["valor"])
    if q["tipo"] == "massa":
        txt = f"{v / 1000:g} kg" if v >= 1000 else f"{v:g} g"
    elif q["tipo"] == "volume":
        txt = f"{v / 1000:g} L" if v >= 1000 else f"{v:g} ml"
    else:
        txt = f"{v:g} unidades"
    if q["pacote"] > 1:
        txt = f"{q['pacote']} x {txt}"
    return f"{texto} | quantidade: {txt}"

In [ ]:
tr = pd.read_parquet(DADOS / "treino.parquet")
va = pd.read_parquet(DADOS / "validacao.parquet")
te = pd.read_parquet(DADOS / "teste.parquet")
cat = json.loads((DADOS / "categorias.json").read_text())
CLASSES = list(cat["chaves"])
K = len(CLASSES)
Y = {c: i for i, c in enumerate(CLASSES)}
if LIMITE:
    tr, va, te = tr.head(LIMITE), va.head(max(LIMITE // 4, 8)), te.head(max(LIMITE // 4, 8))
for d in (tr, va, te):
    d["texto_q"] = [texto_com_quantidade(t) for t in d["texto"]]
print(f"treino {len(tr):,} | validação {len(va):,} | teste {len(te):,} | {K} categorias")
print(te[["texto", "texto_q"]].head(5).to_string())
RES = {"classes": CLASSES, "n_teste": len(te), "modelos": {}}

def salvar():
    (SAIDA / "resultados_v2.json").write_text(json.dumps(RES, indent=1, ensure_ascii=False))

## Treino e resposta

Laço simples em PyTorch: AdamW, aquecimento + cosseno, fp16 com *GradScaler* na T4. No LoRA, os pesos do modelo
ficam congelados em fp16 e só o adapter (r=16 em todas as projeções) e a cabeça de classificação treinam, em fp32.
Ao fim de cada época: acerto na validação; a melhor época guarda as probabilidades da validação e do teste.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

ALVOS_LORA = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def lotes(tok, textos, ys, tam, embaralhar):
    idx = list(range(len(textos)))
    if embaralhar:
        random.shuffle(idx)
    for i in range(0, len(idx), tam):
        parte = idx[i:i + tam]
        enc = tok([textos[j] for j in parte], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        enc = {k: v.to(DISP) for k, v in enc.items()}
        yield enc, (torch.tensor([ys[j] for j in parte], device=DISP) if ys is not None else None)

@torch.no_grad()
def probabilidades(modelo, tok, textos, lote=128):
    modelo.eval()
    out = np.zeros((len(textos), K), dtype=np.float32)
    ordem = np.argsort([len(t) for t in textos])
    for i in range(0, len(textos), lote):
        idx = ordem[i:i + lote]
        enc = tok([textos[j] for j in idx], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(modelo.device)
        with torch.autocast(DISP, dtype=torch.float16, enabled=GPU):
            logits = modelo(**enc).logits.float()
        out[idx] = torch.softmax(logits, -1).cpu().numpy()
    return out

@torch.no_grad()
def latencia(modelo, tok, textos, n):
    """Mediana de n produtos, um por vez (como um cadastro feito na hora), com a tokenização incluída."""
    modelo.eval(); disp = next(modelo.parameters()).device; ts = []
    usa_gpu = disp.type == "cuda"
    for t in textos[:n]:
        if usa_gpu: torch.cuda.synchronize()
        t0 = time.perf_counter()
        enc = tok([t], truncation=True, max_length=MAX_LEN, return_tensors="pt").to(disp)
        with torch.autocast(disp.type, dtype=torch.float16, enabled=usa_gpu):
            modelo(**enc).logits.argmax(-1).item()
        if usa_gpu: torch.cuda.synchronize()
        ts.append(time.perf_counter() - t0)
    return round(1000 * float(np.median(ts)), 2)

def treinar(chave, nome, coluna, lora, epocas, lr, batch=32, cpu_lat=0):
    t_ini = time.time()
    tok = AutoTokenizer.from_pretrained(nome)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    tipo = torch.float16 if (lora and GPU) else torch.float32
    modelo = AutoModelForSequenceClassification.from_pretrained(
        nome, num_labels=K, dtype=tipo, id2label=dict(enumerate(CLASSES)), label2id=Y, ignore_mismatched_sizes=True)
    modelo.config.pad_token_id = tok.pad_token_id
    if lora:
        modelo = get_peft_model(modelo, LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.05,
                                                   bias="none", target_modules=ALVOS_LORA))
    modelo.to(DISP)
    treinaveis = [p for p in modelo.parameters() if p.requires_grad]
    for p in treinaveis:
        p.data = p.data.float()
    n_trein, n_total = sum(p.numel() for p in treinaveis), sum(p.numel() for p in modelo.parameters())
    print(f"{chave}: {n_trein:,} de {n_total:,} parâmetros treinam ({100 * n_trein / n_total:.2f}%)")
    opt = torch.optim.AdamW(treinaveis, lr=lr, weight_decay=0.0 if lora else 0.01)
    passos = epocas * math.ceil(len(tr) / batch)
    aquec = max(1, int(0.06 * passos))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / aquec) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / passos))))
    scaler = torch.amp.GradScaler("cuda", enabled=GPU)
    xtr, ytr = list(tr[coluna]), [Y[c] for c in tr["categoria"]]
    xva, yva, xte = list(va[coluna]), np.array([Y[c] for c in va["categoria"]]), list(te[coluna])
    hist, melhor = [], {"acc": -1}
    t_treino = 0.0
    for ep in range(epocas):
        modelo.train(); soma = n = 0; t0 = time.time()
        for enc, y in lotes(tok, xtr, ytr, batch, True):
            with torch.autocast(DISP, dtype=torch.float16, enabled=GPU):
                logits = modelo(**enc).logits
            perda = torch.nn.functional.cross_entropy(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(perda).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(treinaveis, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            soma += perda.item(); n += 1
        t_treino += time.time() - t0
        pva = probabilidades(modelo, tok, xva)
        acc = float((pva.argmax(1) == yva).mean())
        hist.append({"epoca": ep + 1, "perda_treino": round(soma / max(n, 1), 4), "acerto_validacao": round(acc, 4),
                     "min_treino": round(t_treino / 60, 2)})
        print(chave, hist[-1], flush=True)
        if acc > melhor["acc"]:
            if GPU: torch.cuda.synchronize()
            t0 = time.time(); pte = probabilidades(modelo, tok, xte)
            if GPU: torch.cuda.synchronize()
            melhor = {"acc": acc, "epoca": ep + 1, "pva": pva, "pte": pte, "ms_lote": 1000 * (time.time() - t0) / len(xte)}
    linhas = [("validacao", i, p) for i, p in zip(va["id"], melhor["pva"])] + [("teste", i, p) for i, p in zip(te["id"], melhor["pte"])]
    probs = pd.DataFrame([p for _, _, p in linhas], columns=CLASSES).round(5)
    probs.insert(0, "id", [i for _, i, _ in linhas]); probs.insert(0, "parte", [a for a, _, _ in linhas])
    probs.to_parquet(SAIDA / f"probs_{chave}.parquet", index=False)
    info = {"nome": nome, "coluna": coluna, "lora": lora, "epocas": epocas, "lr": lr, "batch": batch,
            "epoca_escolhida": melhor["epoca"], "acerto_validacao": round(melhor["acc"], 4), "historico": hist,
            "min_treino": round(t_treino / 60, 2), "parametros_treinaveis": int(n_trein), "parametros_total": int(n_total),
            "ms_por_item_lote_gpu": round(melhor["ms_lote"], 3), "ms_um_a_um_gpu": latencia(modelo, tok, xte, 50) if GPU else None}
    if cpu_lat:
        try:
            m_cpu = modelo.merge_and_unload() if lora else modelo
            m_cpu = m_cpu.float().cpu()
            info["ms_um_a_um_cpu"] = latencia(m_cpu, tok, xte, cpu_lat)
            info["cpu_threads"] = torch.get_num_threads()
            del m_cpu
        except Exception as e:
            info["ms_um_a_um_cpu_erro"] = repr(e)[:300]
    info["min_total"] = round((time.time() - t_ini) / 60, 2)
    RES["modelos"][chave] = info
    salvar()
    print(chave, {k: v for k, v in info.items() if k != "historico"}, flush=True)
    del modelo, opt, scaler
    gc.collect()
    if GPU: torch.cuda.empty_cache()

In [ ]:
BERT = os.environ.get("CATALOGO_BERT", "neuralmind/bert-base-portuguese-cased")
QWEN = os.environ.get("CATALOGO_QWEN", "Qwen/Qwen2.5-1.5B-Instruct")
QWEN3B = os.environ.get("CATALOGO_QWEN3B", "Qwen/Qwen2.5-3B-Instruct")
E = 1 if LOCAL else None

treinar("bertimbau_q", BERT, "texto_q", lora=False, epocas=E or 6, lr=5e-5, cpu_lat=30)
treinar("qwen15_lora_q", QWEN, "texto_q", lora=True, epocas=E or 3, lr=2e-4, cpu_lat=10)
treinar("bertimbau_sem_q", BERT, "texto", lora=False, epocas=E or 6, lr=5e-5)

## Extra: Qwen2.5-3B + LoRA (o dobro do tamanho)

Se faltar memória ou tempo, a célula registra o motivo e segue (os outros resultados já estão salvos).

In [ ]:
if not LOCAL and GPU:
    try:
        treinar("qwen3b_lora_q", QWEN3B, "texto_q", lora=True, epocas=3, lr=1e-4, batch=32)
    except Exception as e:
        RES["modelos"]["qwen3b_lora_q"] = {"pulado": repr(e)[:500]}
        print("3B pulado:", repr(e)[:500]); salvar()
salvar()
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "historico"} for k, v in RES["modelos"].items()}, indent=1, ensure_ascii=False))

---

## English summary

**Grocery catalog v2.** The tuned models become **classifiers** (a classification head instead of generating the
category name token by token): one forward pass returns a probability for each of the 41 categories, so it is faster
one item at a time, never answers outside the list, and gives a confidence score that can be combined with the
TF-IDF baseline. The parsed, normalized **quantity** goes into the text (`... | quantidade: 34 g`). Models: BERTimbau
base (full fine-tune, with and without the quantity), Qwen2.5-1.5B + LoRA and, as an extra, Qwen2.5-3B + LoRA, on a
free Kaggle T4. The epoch is picked on **validation**; test metrics are computed outside, by the same script as the
other approaches. Repo: https://github.com/Bueno-00/catalogo-de-mercado